In [24]:
import pandas as pd

In [25]:
from glob import glob
import time
import random

In [26]:
logs = pd.concat([pd.read_json(f, lines=True) for f in glob('/projects/bdata/photocritique-coach-extension/logs/*.jsonl')], axis='rows')
print(f'Loaded {len(logs):d} logs from disk.')

Loaded 1287 logs from disk.


In [27]:
participants = pd.read_csv('/projects/bdata/photocritique-coach-extension/logs/participants.csv')
participants.enrollmentTimestamp = pd.to_datetime(participants.enrollmentTimestamp)
print(f'Loaded {len(participants):d} participants from disk.')

Loaded 13 participants from disk.


In [28]:
participants

,username,email,enrollmentTimestamp,assignedInterface
0,burly-gurly,NaN,2025-09-13 07:47:10.446000+00:00,tutor
1,lew_traveler,llorton@gmail.com,2025-09-14 17:06:40.348000+00:00,tutor
2,_Scorpion_1,adam.tencer@gmail.com,2025-09-14 19:40:41.129000+00:00,control
3,leshlon,malikquess@gmail.com,2025-09-15 07:21:55.390000+00:00,tutor
4,Normal-Armadillo8961,delphiniser@gmail.com,2025-09-23 08:07:06.908000+00:00,tutor
5,noahmaier,maiernoah@gmail.com,2025-09-24 00:47:28.834000+00:00,assistant
6,frootyFYt,drekoray09@gmail.com,2025-09-25 11:04:10.054000+00:00,assistant
7,cyclistNerd,galen.weld@gmail.com,2025-10-27 01:18:13.338000+00:00,assistant
8,yumcax,NaN,2025-11-06 02:03:39.699000+00:00,tutor
9,10pmet,somnaang.rou@gmail.com,2025-11-06 05:15:29.804000+00:00,tutor


# overview

Two primary tasks: first, message people who recently finished their 10 day learning period, and second, send daily 'homework' to active participants.

In [29]:
participants['enrollmentDuration'] = pd.Timestamp.utcnow() - participants.enrollmentTimestamp

In [30]:
active   = participants.enrollmentDuration <= pd.Timedelta(days=10)
finished = (participants.enrollmentDuration > pd.Timedelta(days=10)) & (participants.enrollmentDuration <= pd.Timedelta(days=11))
inactive = participants.enrollmentDuration > pd.Timedelta(days=11)

active   = participants.loc[active,:]
finished = participants.loc[finished,:]
inactive = participants.loc[inactive,:]

print(f'Divided into {len(active):,d} active participants, {len(finished):,d} finished participants, and {len(inactive)} inactive participants.')

Divided into 5 active participants, 0 finished participants, and 8 inactive participants.


## compute comments made using tool

In [31]:
comments = logs.loc[logs.endpoint=='submitted',:].dropna(how='all', axis='columns')

In [32]:
# get earliest enrollment date for participants to join
comments = pd.merge(comments,
         #           using min here but it shouldn't matter since participants should only be assinged one interface 
         participants.groupby('username').agg({'enrollmentTimestamp':'min','assignedInterface':'min'}).reset_index(),
         how='left',
         left_on='currentUsername',
         right_on='username'
        ).drop('currentUsername', axis='columns')

In [33]:
comments['timeSinceEnrollment'] = comments.timestamp-comments.enrollmentTimestamp

In [34]:
comments['activelyEnrolled'] = comments.timeSinceEnrollment <= pd.Timedelta(days=10)

In [35]:
print(f'Loaded {len(comments):,d} comments made by participants, {comments.activelyEnrolled.sum():,d} were made while they were active in the study.')

Loaded 151 comments made by participants, 118 were made while they were active in the study.


In [13]:
# get number of comments made by active users
print('')
print(
    pd.concat([
        active.set_index('username').enrollmentDuration.dt.days.rename('days'),
        comments.loc[comments.activelyEnrolled,:].groupby('username').agg({'timestamp':'count'}).timestamp.rename('comments')
    ], axis='columns').dropna(how='any', subset=['days'], axis='rows').fillna(0).astype('int')
)


                days  comments
username                      
yumcax             4         0
10pmet             3         8
Top-Order-2878     3         3
darktriaddryad     2        42
HCGAdrianHolt      0         2


## get ready to send messages

In [14]:
import sys
sys.path.append('..')
from setup import reddit

Version 7.5.0 of praw is outdated. Version 7.8.1 was released Friday October 25, 2024.


In [15]:
EOS_SUBJECT = 'Thanks for trying Photocritique Coach'

EOS_MSG = \
'''
Thanks for using Photocritique Coach! You've finished your 10 day coaching session, so the extension will automatically
disable itself from your web browser if it hasn't already.

It looks like you wrote {num_comments} comments with help from the coach, thank you!
{flair_yes_no}

We'd love to hear any additional feedback you have about Photocritique Coach.
Please message /u/cyclistNerd if you have comments or questions.
This account can't respond to messages.

You earned {num_tickets} raffle points for participating in our study - but you can earn more!
You can get 1 additional raffle ticket for each comment you write, up to 50 tickets.
**In late-December we'll raffle off a brand new camera of your choice, plus more prizes!**

See [here](https://www.reddit.com/r/photocritique/wiki/browserextension/) for more details on the raffle.
'''

PARTICIPANT_FLAIR_PATH = '/projects/bdata/photocritique-coach-extension/logs/finished_flaired_participants.csv'

# message people who just finished

In [16]:
print(f'Sending messages to {len(finished):,d} users who finished recently...')
for _, row in finished.iterrows():
    num_comments = comments.loc[comments.username==row.username,'activelyEnrolled'].sum()
    print(f'/u/{row.username} has made {num_comments} comments during their active enrollment.', end=' ')
    
    if num_comments<10:
        flair = "Unfortunately this isn't enough comments to earn a star in your flair."
        print('Not Giving Flair.')
    else:
        flair = "As a shout-out for your hard work, you'll get your star in your flair within an hour or so."
        print('Giving Flair.')
        with open(PARTICIPANT_FLAIR_PATH, 'a') as f:
            f.write(f'{row.username},{str(int(time.time()))}\n')
        
    formatted_message=EOS_MSG.format(
        num_comments=num_comments,
        flair_yes_no=flair,
        num_tickets=min(num_comments, 50),
    )
    
    try:
        print('\tSending end of study message.')
        reddit.redditor(row.username).message(subject=EOS_SUBJECT, message=formatted_message)
        
    except Exception as e:
        print('\tError sending message.')
        print(e)
    time.sleep(2)

Sending messages to 0 users who finished recently...


# send 'homework'

In [17]:
# compute posts that have already received a treated comment
posts = comments.loc[comments.activelyEnrolled,:].groupby('postId').agg({'timestamp':'max', 'assignedInterface':set})
posts['most_recent_treatment_activity'] = pd.Timestamp.utcnow() - posts.timestamp
posts = posts.drop('timestamp',axis='columns')

In [18]:
for condition in ['control','assistant','tutor']:
    posts.loc[:,condition] = posts.assignedInterface.apply(lambda l: condition in l)
posts = posts.drop('assignedInterface',axis='columns')

In [19]:
# we only want to suggest posts with some recent activity
posts = posts.loc[posts.most_recent_treatment_activity<=pd.Timedelta(days=5)]
print(f'Filtered to suggesting from {len(posts):,d} posts with a comment within the past 5 days.')

Filtered to suggesting from 50 posts with a comment within the past 5 days.


In [20]:
# get filler posts without many comments that we can use if we don't have enough to make recommendations with
posts_with_few_comments = []

subreddit = reddit.subreddit('photocritique')
for submission in subreddit.new(limit=100):  # can increase if needed
    if submission.num_comments < 5:
        posts_with_few_comments.append(submission)

print(f'Found {len(posts_with_few_comments)} posts with < 5 comments')

Found 25 posts with < 5 comments


In [21]:
# # TODO NEED TO REMOVE!!!!!

# dummy_data = pd.DataFrame([{
#     'username': 'testUser',
#     'email': 'none',
#     'enrollmentTimestamp': pd.Timestamp('2025-11-02 21:28:33.593000+0000', tz='UTC'),
#     'assignedInterface': 'assistant',
#     'enrollmentDuration': pd.Timedelta('3 days 02:25:55.240402')
# }])

# active = pd.concat([active, dummy_data], ignore_index=True)

In [22]:
HOMEWORK_SUBJECT = 'Photocritique Coach: Day {} Reminder!'

HOMEWORK_MSG = \
'''
Hi u/{username}, thanks for trying out Photocritique Coach!

This is your daily reminder to write some comments using the coach.
According to our records, in the {enrolled_days} day{enrolled_days_plural} you've used the coach,
you've written {num_comments} comment{num_comments_plural}. You need to write at least 10 comments
total to get a star in your flair! You have {days_left} day{days_left_plural} days left to use the coach.
For each comment you write, you'll get an additional raffle ticket to **win a camera**!

Here are some posts that we think could use a comment from you!

'''

HOMEWORK_MSG_END = \
'''

If you have any questions or feedback, please reach out to /u/cyclistNerd!
This account can't respond to messages.
'''

def message_homework(username, enrollment_duration, posts_to_assign):
    num_comments = comments.loc[comments.username==username,'activelyEnrolled'].sum()
    
    formatted_message = HOMEWORK_MSG.format(
        username      = username,
        enrolled_days = enrollment_duration.days+1,
        enrolled_days_plural = '' if (enrollment_duration.days+1)==1 else 's',
        num_comments = num_comments,
        num_comments_plural = '' if num_comments==1 else 's',
        days_left = 10-enrollment_duration.days,
        days_left_plural = '' if 10-enrollment_duration.days==1 else 's',
    )
    
    for post in posts_to_assign:
        formatted_message += f'* [{post.title}](https://www.reddit.com/r/{post.permalink})\n'
    
    formatted_message += HOMEWORK_MSG_END
    
    reddit.redditor(username).message(
        subject=HOMEWORK_SUBJECT.format(enrollment_duration.days),
        message=formatted_message
    )
    return

In [23]:
print(f'\nSending homework to {len(active)} active participants.')
for _, participant in active.iterrows():
    print(f'/u/{participant.username} ({participant.assignedInterface}) has been enrolled {participant.enrollmentDuration.days} days...')
    
    # get recent posts that no one in this condition has commented on
    eligible_posts = posts.index[~posts.loc[:,participant.assignedInterface]]
    print(f'\tSelecting from {len(eligible_posts)} eligible posts...')
    
    posts_to_assign = [reddit.submission(post_id) for post_id in random.sample(eligible_posts.to_list(), k=min(len(eligible_posts), 3))]
    
    if len(posts_to_assign) < 3:
        needed = 3 - len(posts_to_assign)
        # Make sure we don't try to sample more than available
        available = min(needed, len(posts_with_few_comments))
        print(f'\tAdding {available} posts with few comments.')
        posts_to_assign += random.sample(posts_with_few_comments, k=available)
        
    # send messages
    try:
        print(f'\tMessaging u/{participant.username} with {len(posts_to_assign)} posts')
        message_homework(participant.username, participant.enrollmentDuration, posts_to_assign)
        
    except Exception as e:
        print('\tError sending homework:')
        print(e)
    
    finally:
        time.sleep(2)


Sending homework to 5 active participants.
/u/yumcax (tutor) has been enrolled 4 days...
	Selecting from 3 eligible posts...
	Messaging u/yumcax with 3 posts
/u/10pmet (tutor) has been enrolled 3 days...
	Selecting from 3 eligible posts...
	Messaging u/10pmet with 3 posts
/u/Top-Order-2878 (control) has been enrolled 3 days...
	Selecting from 47 eligible posts...
	Messaging u/Top-Order-2878 with 3 posts
/u/darktriaddryad (tutor) has been enrolled 2 days...
	Selecting from 3 eligible posts...
	Messaging u/darktriaddryad with 3 posts
/u/HCGAdrianHolt (tutor) has been enrolled 0 days...
	Selecting from 3 eligible posts...
	Messaging u/HCGAdrianHolt with 3 posts
